In [1]:
import json
import warnings
from pathlib import Path

from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter('ignore', InterpolationWarning)

import config
from input.input import load_raw_data
from model import generics, single_ml_model_exp, grid_search_exp
from model.feature_selection import TimeSeriesFeatureSelector
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from utils.compare_fs_vs_baseline import build_comparison
from utils.export_metrics_to_csv import save_csv

%load_ext autoreload
%autoreload 2

Failed to read module file 'C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\re\_casefix.py' for module 're._casefix': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  File "<frozen i

In [2]:
# === Notebook de FS na janela de 10% (pct10) -- MLP single (SKlearnModel) / f_test ===
# Mesma estrutura dos chamados_v4_fs_* ('auto'), mas: (a) lag_size_override =
# resolve_lag_size_pct(N - test_size, 0.10) por serie; (b) comparacao par-a-par
# SEMPRE contra o baseline pct10 DA PROPRIA FAMILIA (celula final), NUNCA
# contra o baseline 'auto'. experiment_id/model_name distintos, sem underscore.
# seed=42 (familia MLP). force=False. Unica acao: Restart Kernel -> Run All.
model = Pipeline([
    ('selector', TimeSeriesFeatureSelector(strategy='f_test')),
    ('estimator', MLPRegressor(activation='logistic', solver='lbfgs')),
])

series_list = ['airlines.txt', 'austres.txt', 'coloradoRiver.txt', 'sunspot.txt', 'windspeedfortaleza.txt', 'samurec.txt']

experiment_id = 'chamados_pct10_fs_mlp_ftest'
model_name = 'mlppct10ftest'   # -> 1mlppct10ftest.pkl (sem underscore, RUNBOOK.md 7)
normalize = True
force = False
model_exec = 10

experiment_params = {
    'diff_kpss': False,
    'horizon': 1,
    'type_filter': None,
}

model_parameters = {
    'selector__k': [1, 5, 9, 15, 20],
    'estimator__hidden_layer_sizes': [10, 20, 50],
    'estimator__max_iter': [1000],
}

# Comparacao par-a-par: baseline pct10 da PROPRIA familia (Parte 1). Nunca 'auto'.
baseline_experiment_id = 'chamados_pct10'
baseline_model_name = '1mlppct10'
linear_model_name_to_exclude = None

experiment_dir = Path(config.MODEL_DATA_PATH) / experiment_id
experiment_dir_results = Path(config.ROOT_PATH) / 'results' / experiment_id

In [3]:
# Sanity-check (mesmo padrao dos notebooks 'auto'): Pipeline.get_params(deep=True)
# expoe as chaves que GridSearch vai usar; e strategy <-> experiment_id/model_name
# consistentes entre si.
params = model.get_params(deep=True)
required_keys = {'selector__strategy', 'selector__k', 'estimator__hidden_layer_sizes', 'estimator__max_iter'}
missing = required_keys - params.keys()
assert not missing, f'get_params(deep=True) nao expos: {missing}'

strategy_slug = model.named_steps['selector'].strategy.replace('_', '')
assert strategy_slug in experiment_id, f'{strategy_slug!r} nao em experiment_id={experiment_id!r}'
assert strategy_slug in model_name, f'{strategy_slug!r} nao em model_name={model_name!r}'
print(f'OK -- strategy={model.named_steps["selector"].strategy!r} consistente; keys expostas.')

OK -- strategy='f_test' consistente; keys expostas.


In [4]:
# Janela de 10%: lag_size_override = resolve_lag_size_pct(N - test_size, 0.10)
# por serie -- MESMA base que get_max_lag_to_consider (PACF sobre
# ts_univariate[0:-test_size]). Computado aqui, nada a editar.
# force=False: execution() (correcao de 2026-09-02) pula .pkl ja existente
# e nao-vazio -- re-Run All e idempotente.
lag_pct_por_serie = {}
for base_name in series_list:
    n_raw = len(load_raw_data(base_name))
    n_train = n_raw - int(config.TEST_SIZE * n_raw)
    lag_pct = grid_search_exp.resolve_lag_size_pct(n_train, pct=0.10)
    lag_pct_por_serie[base_name] = lag_pct
    print(f'{base_name}  N={n_raw}  N-test={n_train}  lag_pct={lag_pct}')
    exec_gs = grid_search_exp.GridSearch(
        single_ml_model_exp.SKlearnModel,
        model,
        model_parameters,
        experiment_id,
        base_name,
        model_name,
        force,
        normalize,
        experiment_params,
        model_exec=model_exec,
        use_val_slipt_for_prev=True,
        lag_size_override=lag_pct,
        estimator_random_state_base=grid_search_exp.MLP_RANDOM_STATE_BASE,  # CLAUDE.md 3.4 -- seed fixa das familias MLP
    )
    exec_gs.execution()

airlines.txt  N=144  N-test=130  lag_pct=13
{'estimator__hidden_layer_sizes': 10, 'estimator__max_iter': 1000, 'selector__k': 5}
austres.txt  N=89  N-test=81  lag_pct=8
{'estimator__hidden_layer_sizes': 50, 'estimator__max_iter': 1000, 'selector__k': 1}
coloradoRiver.txt  N=744  N-test=670  lag_pct=67
{'estimator__hidden_layer_sizes': 50, 'estimator__max_iter': 1000, 'selector__k': 9}
sunspot.txt  N=288  N-test=260  lag_pct=26
{'estimator__hidden_layer_sizes': 50, 'estimator__max_iter': 1000, 'selector__k': 9}
windspeedfortaleza.txt  N=144  N-test=130  lag_pct=13
{'estimator__hidden_layer_sizes': 20, 'estimator__max_iter': 1000, 'selector__k': 15}
samurec.txt  N=1188  N-test=1070  lag_pct=107
{'estimator__hidden_layer_sizes': 10, 'estimator__max_iter': 1000, 'selector__k': 20}


In [5]:
from utils.export_metrics_to_csv import run_export_metrics_to_csv

df_metrics = run_export_metrics_to_csv(
    experiment_dir, experiment_dir_results / 'metrics.csv', detail=True,
)
df_metrics

[INFO] 6 arquivo(s) .pkl encontrado(s) em 'C:\Projetos\mestrado_codigos\experiments\data\result\chamados_pct10_fs_mlp_ftest'.

  OK  airlines_1mlppct10ftest.pkl  ->  10 linha(s)
  OK  austres_1mlppct10ftest.pkl  ->  10 linha(s)
  OK  coloradoRiver_1mlppct10ftest.pkl  ->  10 linha(s)
  OK  samurec_1mlppct10ftest.pkl  ->  10 linha(s)
  OK  sunspot_1mlppct10ftest.pkl  ->  10 linha(s)
  OK  windspeedfortaleza_1mlppct10ftest.pkl  ->  10 linha(s)

[OK] CSV agregado (média das repetições) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_pct10_fs_mlp_ftest\metrics.csv
     6 linha(s) × 20 coluna(s)

[OK] CSV detalhado (por repetição) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_pct10_fs_mlp_ftest\metrics_detail.csv
     60 linha(s) × 13 coluna(s)


,ExperimentID,Serie,Modelo,N_Repeticoes,MSE_mean,MSE_std,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,theil_mean,theil_std,ARV_mean,ARV_std,IA_mean,IA_std,POCID_mean,POCID_std
0,chamados_pct10_fs_mlp_ftest,airlines,1mlppct10ftest,10,481.974625,150.975021,21.745773,3.179099,16.932624,2.509430,3.632590,0.454129,0.278647,0.203195,0.099624,0.043180,0.976974,0.008665,80.000000,3.011693
1,chamados_pct10_fs_mlp_ftest,austres,1mlppct10ftest,10,657.878853,840.741105,22.239701,13.469082,19.868996,14.063995,0.113490,0.080262,0.349751,0.459920,0.059542,0.071175,0.983689,0.020567,87.500000,0.000000
2,chamados_pct10_fs_mlp_ftest,coloradoRiver,1mlppct10ftest,10,0.042574,0.001490,0.206307,0.003612,0.168734,0.003070,19.675070,0.352533,1.427453,0.069057,0.628846,0.010876,0.780047,0.005931,59.054054,1.282004
3,chamados_pct10_fs_mlp_ftest,samurec,1mlppct10ftest,10,46.203658,0.373466,6.797277,0.027531,5.556994,0.034757,19.044094,0.110974,8.844418,0.222578,13.064494,0.356573,0.346914,0.007657,58.898305,1.560079
4,chamados_pct10_fs_mlp_ftest,sunspot,1mlppct10ftest,10,291.471228,6.282232,17.071655,0.182088,13.292740,0.127300,33.459311,0.709157,0.309282,0.007000,0.165731,0.003478,0.960072,0.000775,67.857143,0.000000
5,chamados_pct10_fs_mlp_ftest,windspeedfortaleza,1mlppct10ftest,10,0.135257,0.003981,0.367736,0.005495,0.279743,0.002939,9.524331,0.104173,0.544635,0.003232,0.225732,0.007188,0.938805,0.001973,65.000000,2.258770


In [6]:
from utils.export_selected_features import run_export_selected_features

df_features = run_export_selected_features(
    experiment_dir, experiment_dir_results / 'selected_features.csv', detail=True,
)
df_features

[INFO] 6 arquivo(s) .pkl encontrado(s) em 'C:\Projetos\mestrado_codigos\experiments\data\result\chamados_pct10_fs_mlp_ftest'.

  OK  airlines_1mlppct10ftest.pkl  ->  10 linha(s)
  OK  austres_1mlppct10ftest.pkl  ->  10 linha(s)
  OK  coloradoRiver_1mlppct10ftest.pkl  ->  10 linha(s)
  OK  samurec_1mlppct10ftest.pkl  ->  10 linha(s)
  OK  sunspot_1mlppct10ftest.pkl  ->  10 linha(s)
  OK  windspeedfortaleza_1mlppct10ftest.pkl  ->  10 linha(s)

[OK] CSV agregado (média/desvio por série × modelo) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_pct10_fs_mlp_ftest\selected_features.csv
     6 linha(s) × 8 coluna(s)

[OK] CSV detalhado (por repetição) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_pct10_fs_mlp_ftest\selected_features_detail.csv
     60 linha(s) × 9 coluna(s)


,ExperimentID,Serie,Modelo,Strategy,N_Features_Selected_mean,N_Features_Selected_std,N_Repeticoes,N_Features_Total
0,chamados_pct10_fs_mlp_ftest,airlines,1mlppct10ftest,f_test,5.0,0.0,10,13
1,chamados_pct10_fs_mlp_ftest,austres,1mlppct10ftest,f_test,1.0,0.0,10,8
2,chamados_pct10_fs_mlp_ftest,coloradoRiver,1mlppct10ftest,f_test,9.0,0.0,10,67
3,chamados_pct10_fs_mlp_ftest,samurec,1mlppct10ftest,f_test,20.0,0.0,10,107
4,chamados_pct10_fs_mlp_ftest,sunspot,1mlppct10ftest,f_test,9.0,0.0,10,26
5,chamados_pct10_fs_mlp_ftest,windspeedfortaleza,1mlppct10ftest,f_test,13.0,0.0,10,13


In [7]:
# Comparacao PAR-A-PAR: FS pct10 x baseline pct10 da MESMA familia.
# NUNCA contra o baseline 'auto' -- isola o efeito da selecao de features
# do efeito da definicao de janela. A chave do dict e so o rotulo da coluna
# de saida (o slug da estrategia).
fs_dirs = {experiment_id.rsplit('_', 1)[-1]: experiment_dir}
df_cmp = build_comparison(
    Path(config.MODEL_DATA_PATH) / baseline_experiment_id,
    fs_dirs,
    baseline_model_name=baseline_model_name,
    linear_model_name_to_exclude=linear_model_name_to_exclude,
)
save_csv(df_cmp, experiment_dir_results / 'comparison.csv',
         label='comparacao FS pct10 x baseline pct10 (par-a-par)')
df_cmp

[INFO] 30 arquivo(s) .pkl encontrado(s) em 'C:\Projetos\mestrado_codigos\experiments\data\result\chamados_pct10'.

  OK  airlines_1amv1pct10.pkl  ->  10 linha(s)
  OK  airlines_1arima.pkl  ->  1 linha(s)
  OK  airlines_1aspct10.pkl  ->  1 linha(s)
  OK  airlines_1mlppct10.pkl  ->  10 linha(s)
  OK  airlines_1svrpct10.pkl  ->  1 linha(s)
  OK  austres_1amv1pct10.pkl  ->  10 linha(s)
  OK  austres_1arima.pkl  ->  1 linha(s)
  OK  austres_1aspct10.pkl  ->  1 linha(s)
  OK  austres_1mlppct10.pkl  ->  10 linha(s)
  OK  austres_1svrpct10.pkl  ->  1 linha(s)
  OK  coloradoRiver_1amv1pct10.pkl  ->  10 linha(s)
  OK  coloradoRiver_1arima.pkl  ->  1 linha(s)
  OK  coloradoRiver_1aspct10.pkl  ->  1 linha(s)
  OK  coloradoRiver_1mlppct10.pkl  ->  10 linha(s)
  OK  coloradoRiver_1svrpct10.pkl  ->  1 linha(s)
  OK  samurec_1amv1pct10.pkl  ->  10 linha(s)
  OK  samurec_1arima.pkl  ->  1 linha(s)
  OK  samurec_1aspct10.pkl  ->  1 linha(s)
  OK  samurec_1mlppct10.pkl  ->  10 linha(s)
  OK  samurec_1svr

,Serie,Baseline_RMSE,ftest_RMSE,ftest_PctGain,ftest_NFeatures
0,airlines,28.856466,21.745773,24.641596,5.0
1,austres,24.680016,22.239701,9.887820,1.0
2,coloradoRiver,0.210362,0.206307,1.927563,9.0
3,samurec,7.187309,6.797277,5.426676,20.0
4,sunspot,19.222508,17.071655,11.189246,9.0
5,windspeedfortaleza,0.367736,0.367736,0.000000,13.0


In [8]:
import json

metadata = {
    'experiment_id': experiment_id,
    'notebook': 'single_models/mlp_pct10_ftest.ipynb',
    'tipo': 'fs pct10',
    'familia': 'MLP single (SKlearnModel) / f_test',
    'janela': 'pct10 -- resolve_lag_size_pct(N - int(config.TEST_SIZE*N), 0.10)',
    'lag_pct_por_serie': {s: lag_pct_por_serie[s] for s in series_list},
    'series': series_list,
    'model_exec': model_exec,
    'seed': grid_search_exp.MLP_RANDOM_STATE_BASE,
    'model_parameters': model_parameters,
    'diff_kpss': experiment_params['diff_kpss'],
    'baseline_pareado': baseline_model_name,
    'strategy': 'f_test',
}
experiment_dir_results.mkdir(parents=True, exist_ok=True)
(experiment_dir_results / 'metadata.json').write_text(
    json.dumps(metadata, indent=2, default=str), encoding='utf-8'
)
print('metadata.json ->', experiment_dir_results / 'metadata.json')

Failed to read module file 'C:\Projetos\mestrado_codigos\experiments\src\model\hybrid_system_exp.py' for module 'model.hybrid_system_exp': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 219, in update_sources
    self.source_by_modname[new_modname] = f.read()
                                          ^^^^^^^^
  File "C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 39146: character maps to <undefined>


metadata.json -> C:\Projetos\mestrado_codigos\experiments\results\chamados_pct10_fs_mlp_ftest\metadata.json
